# Local Common Crawl Sampling and Deduplication

This notebook is designed for local development. It first streams a small sample out of raw Common Crawl WET gzip files, then runs exact-dedup and MinHash on the sampled JSONL instead of parsing the full `.gz` payloads inside Spark.

## Setup and sampling parameters

This section imports the Spark helpers used later, sets the local file paths, and defines the limits that keep the notebook laptop-friendly. Adjust these constants first when you want to trade speed for broader coverage.

In [1]:
from pathlib import Path
import gzip
import json

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.ml.feature import RegexTokenizer, NGram, HashingTF, MinHashLSH

In [2]:
DATA_DIR = Path("data/wet_samples")
LOCAL_SAMPLE_PATH = Path("data/local_wet_sample.jsonl")
FINAL_PARQUET_PATH = Path("data/parquet/final_docs")
DUPLICATE_AUDIT_PATH = Path("data/parquet/duplicate_audit")
CRAWL_ID = "CC-MAIN-2026-08"

MAX_FILES = 10
MAX_DOCS_PER_FILE = 300
MIN_TEXT_CHARS = 200
MAX_TEXT_CHARS = 4_000
MAX_DOCS_FOR_MINHASH = 2_000
JACCARD_DISTANCE_THRESHOLD = 0.25
NGRAM_SIZE = 2
SPARK_DRIVER_MEMORY = "8g"

In [3]:
wet_files = sorted(DATA_DIR.glob("*.gz"))[:MAX_FILES]

if not wet_files:
    raise FileNotFoundError(f"No WET gzip files found under {DATA_DIR}")

print(f"Using {len(wet_files)} WET files:")
for path in wet_files:
    print(" -", path.name)

Using 10 WET files:
 - CC-MAIN-20260207002457-20260207032457-00195.warc.wet.gz
 - CC-MAIN-20260207032847-20260207062847-00278.warc.wet.gz
 - CC-MAIN-20260208095500-20260208125500-00434.warc.wet.gz
 - CC-MAIN-20260208125845-20260208155845-00592.warc.wet.gz
 - CC-MAIN-20260209011130-20260209041130-00289.warc.wet.gz
 - CC-MAIN-20260210105610-20260210135610-00256.warc.wet.gz
 - CC-MAIN-20260210200815-20260210230815-00098.warc.wet.gz
 - CC-MAIN-20260211082311-20260211112311-00048.warc.wet.gz
 - CC-MAIN-20260217073908-20260217103908-00810.warc.wet.gz
 - CC-MAIN-20260218231914-20260219021914-00530.warc.wet.gz


## Build a local JSONL sample

The raw WET files are streamed record by record and converted into a much smaller JSONL file. This keeps local iteration fast and avoids forcing Spark to parse full gzip payloads during every experiment.

In [4]:
def emit_record(lines, source_file, min_chars=MIN_TEXT_CHARS, max_chars=MAX_TEXT_CHARS):
    if not lines:
        return None

    url = None
    warc_type = None
    body_started = False
    body_lines = []

    for line in lines:
        if not body_started:
            if line == "":
                body_started = True
                continue

            if line.startswith("WARC-Type:"):
                warc_type = line.split(":", 1)[1].strip()
            elif line.startswith("WARC-Target-URI:"):
                url = line.split(":", 1)[1].strip()
        else:
            body_lines.append(line)

    # Keep only converted text payloads so local development stays focused on usable body text.
    if warc_type != "conversion" or not url:
        return None

    text = "\n".join(body_lines).strip()
    if len(text) < min_chars:
        return None

    text = text[:max_chars]

    return {
        "source_file": source_file,
        "url": url,
        "text": text,
        "text_len": len(text),
    }


def build_local_sample(paths, output_path, max_docs_per_file=MAX_DOCS_PER_FILE):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    rows_written = 0
    per_file_counts = {}

    with output_path.open("w", encoding="utf-8") as dst:
        for path in paths:
            docs_written = 0
            record_lines = []

            with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as src:
                for raw_line in src:
                    line = raw_line.rstrip("\n")

                    if line.startswith("WARC/1.0"):
                        row = emit_record(record_lines, source_file=path.name)
                        if row is not None:
                            dst.write(json.dumps(row, ensure_ascii=False) + "\n")
                            docs_written += 1
                            rows_written += 1
                            if docs_written >= max_docs_per_file:
                                break
                        record_lines = [line]
                    else:
                        record_lines.append(line)
                else:
                    row = emit_record(record_lines, source_file=path.name)
                    if row is not None and docs_written < max_docs_per_file:
                        dst.write(json.dumps(row, ensure_ascii=False) + "\n")
                        docs_written += 1
                        rows_written += 1

            per_file_counts[path.name] = docs_written

    return per_file_counts, rows_written


per_file_counts, total_rows = build_local_sample(wet_files, LOCAL_SAMPLE_PATH)
print(f"Wrote {total_rows} sampled documents to {LOCAL_SAMPLE_PATH}")
per_file_counts

Wrote 3000 sampled documents to data/local_wet_sample.jsonl


{'CC-MAIN-20260207002457-20260207032457-00195.warc.wet.gz': 300,
 'CC-MAIN-20260207032847-20260207062847-00278.warc.wet.gz': 300,
 'CC-MAIN-20260208095500-20260208125500-00434.warc.wet.gz': 300,
 'CC-MAIN-20260208125845-20260208155845-00592.warc.wet.gz': 300,
 'CC-MAIN-20260209011130-20260209041130-00289.warc.wet.gz': 300,
 'CC-MAIN-20260210105610-20260210135610-00256.warc.wet.gz': 300,
 'CC-MAIN-20260210200815-20260210230815-00098.warc.wet.gz': 300,
 'CC-MAIN-20260211082311-20260211112311-00048.warc.wet.gz': 300,
 'CC-MAIN-20260217073908-20260217103908-00810.warc.wet.gz': 300,
 'CC-MAIN-20260218231914-20260219021914-00530.warc.wet.gz': 300}

In [5]:
if "spark" in globals():
    spark.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("common_crawl_deduplication_local")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/25 21:15:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load, filter, and exact-deduplicate

This section reads the JSONL sample into Spark, drops obviously unwanted rows, and creates a stronger normalized text field for exact duplicate checks. The goal is to remove clear copies before the more expensive near-duplicate stage.

In [6]:
docs_df = spark.read.json(str(LOCAL_SAMPLE_PATH))

docs_df = (
    docs_df
    .withColumn("domain", F.regexp_extract("url", r"https?://([^/]+)", 1))
    .filter(F.col("text_len") >= MIN_TEXT_CHARS)
    .filter(~F.col("url").rlike("(?i).*(porn|xxx|adult).*"))
)

print("rows:", docs_df.count())
docs_df.groupBy("source_file").count().orderBy("source_file").show(truncate=False)

rows: 2985
+-------------------------------------------------------+-----+
|source_file                                            |count|
+-------------------------------------------------------+-----+
|CC-MAIN-20260207002457-20260207032457-00195.warc.wet.gz|299  |
|CC-MAIN-20260207032847-20260207062847-00278.warc.wet.gz|298  |
|CC-MAIN-20260208095500-20260208125500-00434.warc.wet.gz|298  |
|CC-MAIN-20260208125845-20260208155845-00592.warc.wet.gz|299  |
|CC-MAIN-20260209011130-20260209041130-00289.warc.wet.gz|299  |
|CC-MAIN-20260210105610-20260210135610-00256.warc.wet.gz|299  |
|CC-MAIN-20260210200815-20260210230815-00098.warc.wet.gz|298  |
|CC-MAIN-20260211082311-20260211112311-00048.warc.wet.gz|299  |
|CC-MAIN-20260217073908-20260217103908-00810.warc.wet.gz|300  |
|CC-MAIN-20260218231914-20260219021914-00530.warc.wet.gz|296  |
+-------------------------------------------------------+-----+



In [7]:
docs_clean = (
    docs_df
    .select("source_file", "domain", "url", "text")
    # This normalization is intentionally stronger than a display-friendly clean so near-duplicate pages collide more often.
    .withColumn("text_norm", F.lower(F.col("text")))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"https?://\\S+", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"www\\.\\S+", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"[^a-z0-9\\s]", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"\\b\\d+\\b", " "))
    .withColumn("text_norm", F.regexp_replace("text_norm", r"\s+", " "))
    .withColumn("text_norm", F.trim("text_norm"))
    .withColumn("text_norm", F.substring("text_norm", 1, MAX_TEXT_CHARS))
    .filter(F.length("text_norm") >= MIN_TEXT_CHARS)
)

exact_duplicate_examples = (
    docs_clean
    .groupBy("text_norm")
    .agg(
        F.count("*").alias("doc_count"),
        F.collect_list(F.struct("source_file", "url")).alias("docs"),
    )
    .filter(F.col("doc_count") > 1)
    .orderBy(F.desc("doc_count"), F.asc("text_norm"))
)

before_exact = docs_clean.count()
exact_dedup = docs_clean.dropDuplicates(["text_norm"]).cache()
after_exact = exact_dedup.count()

print("before exact dedup:", before_exact)
print("after exact dedup:", after_exact)
print("exact duplicate groups:", exact_duplicate_examples.count())
exact_duplicate_examples.select("doc_count", "docs").show(10, truncate=100)

before exact dedup: 2540
after exact dedup: 2531


exact duplicate groups: 9
+---------+----------------------------------------------------------------------------------------------------+
|doc_count|                                                                                                docs|
+---------+----------------------------------------------------------------------------------------------------+
|        2|[{CC-MAIN-20260207002457-20260207032457-00195.warc.wet.gz, http://clanv.bplaced.net/forum/ucp.php...|
|        2|[{CC-MAIN-20260207032847-20260207062847-00278.warc.wet.gz, http://datos.cide.edu/handle/10089/368...|
|        2|[{CC-MAIN-20260208125845-20260208155845-00592.warc.wet.gz, http://flood.umd.edu/realtime-TMPA.php...|
|        2|[{CC-MAIN-20260207002457-20260207032457-00195.warc.wet.gz, http://forix.autosport-atlas.com/login...|
|        2|[{CC-MAIN-20260209011130-20260209041130-00289.warc.wet.gz, http://fun-und-witze.de/wbb2/jgs_db.ph...|
|        2|[{CC-MAIN-20260208125845-20260208155845-00592.warc.wet.gz, 

## MinHash near-duplicate detection

The goal here is to compare a bounded local sample, not the full raw crawl. We keep the candidate set small so `approxSimilarityJoin` stays laptop-friendly, but use a slightly looser threshold and more forgiving text normalization so near-duplicates show up more clearly in local experiments.

In [8]:
minhash_input = (
    exact_dedup
    .orderBy("source_file", "url")
    .limit(MAX_DOCS_FOR_MINHASH)
    .withColumn("doc_id", F.monotonically_increasing_id())
)

tokenizer = RegexTokenizer(
    inputCol="text_norm",
    outputCol="tokens",
    pattern=r"\W+",
    gaps=True,
    minTokenLength=2,
)

ngram = NGram(n=NGRAM_SIZE, inputCol="tokens", outputCol="shingles")
hashing_tf = HashingTF(
    inputCol="shingles",
    outputCol="features",
    numFeatures=1 << 16,
    binary=True,
)

tokenized = tokenizer.transform(minhash_input)
shingled = ngram.transform(tokenized).filter(F.size("shingles") > 0)
features = (
    hashing_tf.transform(shingled)
    .select("doc_id", "source_file", "domain", "url", "text_norm", "features")
    .cache()
)

print("docs going into MinHash:", features.count())

docs going into MinHash: 2000


In [9]:
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=4)
model = mh.fit(features)

pairs = (
    model.approxSimilarityJoin(features.alias("a"), features.alias("b"), JACCARD_DISTANCE_THRESHOLD, distCol="jaccard_distance")
    .select(
        F.col("datasetA.doc_id").alias("doc_id_a"),
        F.col("datasetB.doc_id").alias("doc_id_b"),
        F.col("datasetA.source_file").alias("source_file_a"),
        F.col("datasetB.source_file").alias("source_file_b"),
        F.col("datasetA.domain").alias("domain_a"),
        F.col("datasetB.domain").alias("domain_b"),
        F.col("datasetA.url").alias("url_a"),
        F.col("datasetB.url").alias("url_b"),
        F.col("jaccard_distance"),
        (F.lit(1.0) - F.col("jaccard_distance")).alias("jaccard_similarity"),
        F.substring(F.col("datasetA.text_norm"), 1, 140).alias("preview_a"),
        F.substring(F.col("datasetB.text_norm"), 1, 140).alias("preview_b"),
    )
    .filter(F.col("doc_id_a") < F.col("doc_id_b"))
    .orderBy(F.col("jaccard_distance").asc(), F.col("url_a").asc())
)

print("candidate near-duplicate pairs:", pairs.count())
pairs.show(50, truncate=80)

candidate near-duplicate pairs: 58
+--------+--------+-------------------------------------------------------+-------------------------------------------------------+-----------------------------+----------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------+------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|doc_id_a|doc_id_b|                                          source_file_a|                                          source_file_b|                     domain_a|                          domain_b|                                                                           url_a|                                                                           url_b|    jaccard_distance|jaccard_similarity|      

In [10]:
best_match_window = Window.partitionBy("doc_id_b").orderBy(
    F.col("jaccard_distance").asc(),
    F.col("url_a").asc(),
)

duplicate_matches = (
    pairs
    # Keep the single strongest earlier match for each removed document so the audit table stays readable.
    .withColumn("pair_rank", F.row_number().over(best_match_window))
    .filter(F.col("pair_rank") == 1)
    .cache()
)

duplicate_ids = duplicate_matches.select(F.col("doc_id_b").alias("doc_id")).distinct()
deduped_docs = minhash_input.join(duplicate_ids, on="doc_id", how="left_anti").cache()

removed_docs = (
    minhash_input.alias("removed")
    .join(duplicate_matches.alias("match"), F.col("removed.doc_id") == F.col("match.doc_id_b"))
    .select(
        F.col("match.jaccard_similarity"),
        F.col("match.source_file_a").alias("kept_source_file"),
        F.col("match.url_a").alias("kept_url"),
        F.col("removed.source_file").alias("removed_source_file"),
        F.col("removed.url").alias("removed_url"),
        F.substring(F.col("removed.text_norm"), 1, 140).alias("removed_preview"),
    )
    .orderBy(F.col("jaccard_similarity").desc(), F.col("removed_url").asc())
)

print("near-duplicate docs removed:", removed_docs.count())
removed_docs.show(20, truncate=80)
print("docs after MinHash-based filtering:", deduped_docs.count())
deduped_docs.select("source_file", "domain", "url").show(20, truncate=80)

near-duplicate docs removed: 38
+------------------+-------------------------------------------------------+--------------------------------------------------------------------------------+-------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|jaccard_similarity|                                       kept_source_file|                                                                        kept_url|                                    removed_source_file|                                                                     removed_url|                                                                 removed_preview|
+------------------+-------------------------------------------------------+--------------------------------------------------------------------------------+-------------------------------------------------------+---

In [11]:
similarity_bands = (
    pairs
    .withColumn(
        "similarity_band",
        F.when(F.col("jaccard_similarity") >= 0.95, F.lit("0.95-1.00"))
        .when(F.col("jaccard_similarity") >= 0.90, F.lit("0.90-0.95"))
        .when(F.col("jaccard_similarity") >= 0.85, F.lit("0.85-0.90"))
        .otherwise(F.lit("0.75-0.85")),
    )
    .groupBy("similarity_band")
    .agg(F.count("*").alias("pair_count"))
    .orderBy(F.desc("similarity_band"))
)

duplicate_domains = (
    removed_docs
    .withColumn("removed_domain", F.regexp_extract("removed_url", r"https?://([^/]+)", 1))
    .groupBy("removed_domain")
    .agg(F.count("*").alias("removed_docs"))
    .orderBy(F.desc("removed_docs"), F.asc("removed_domain"))
)

cross_domain_pairs = pairs.filter(F.col("domain_a") != F.col("domain_b")).count()

print("cross-domain near-duplicate pairs:", cross_domain_pairs)
similarity_bands.show(truncate=False)
duplicate_domains.show(20, truncate=False)

cross-domain near-duplicate pairs: 35
+---------------+----------+
|similarity_band|pair_count|
+---------------+----------+
|0.95-1.00      |5         |
|0.90-0.95      |9         |
|0.85-0.90      |25        |
|0.75-0.85      |19        |
+---------------+----------+

+----------------------------------+------------+
|removed_domain                    |removed_docs|
+----------------------------------+------------+
|connect.releasewire.com           |3           |
|dados.prefeitura.sp.gov.br        |3           |
|031tt.com                         |1           |
|070cc.com                         |1           |
|105ll.com                         |1           |
|198hh.com                         |1           |
|295ss.com                         |1           |
|301777.com                        |1           |
|949ll.com                         |1           |
|aa970.com                         |1           |
|ac-mini-splits.net                |1           |
|airconditionerredwoodcity.co

## Export final Parquet datasets

The final kept documents are written as a Parquet dataset partitioned by `crawl_date`, and the chosen MinHash matches are written to a separate audit dataset. Together these give you a clean downstream table plus traceability for what was removed.

In [12]:
scored_features = (
    model.transform(features)
    .withColumn(
        "minhash_signature_json",
        F.to_json(F.transform("hashes", lambda x: vector_to_array(x))),
    )
    .select("doc_id", "minhash_signature_json")
)

final_docs = (
    deduped_docs.alias("d")
    .join(docs_df.select("source_file", "url", "text_len").alias("raw"), on=["source_file", "url"], how="left")
    .join(scored_features.alias("s"), on="doc_id", how="left")
    .withColumn("crawl_id", F.lit(CRAWL_ID))
    .withColumn("file_timestamp_raw", F.regexp_extract("source_file", r"CC-MAIN-(\\d{14})-", 1))
    .withColumn(
        "crawl_file_timestamp",
        # Some local files may not follow the expected timestamp pattern, so parse failures should become NULL instead of aborting the write.
        F.expr("try_to_timestamp(nullif(file_timestamp_raw, ''), 'yyyyMMddHHmmss')"),
    )
    .withColumn("crawl_date", F.to_date("crawl_file_timestamp"))
    .withColumn("minhash_num_tables", F.lit(4))
    .withColumn("dedup_method", F.lit("exact_plus_minhash"))
    .withColumn("record_status", F.lit("kept"))
    .select(
        "crawl_id",
        "crawl_date",
        "crawl_file_timestamp",
        "doc_id",
        "source_file",
        "domain",
        "url",
        "text",
        "text_len",
        "text_norm",
        "minhash_signature_json",
        "minhash_num_tables",
        "dedup_method",
        "record_status",
    )
)

duplicate_audit = (
    duplicate_matches
    .withColumn("crawl_id", F.lit(CRAWL_ID))
    .withColumn("match_rank", F.lit(1))
    .select(
        "crawl_id",
        "doc_id_a",
        "doc_id_b",
        "source_file_a",
        "source_file_b",
        "domain_a",
        "domain_b",
        "url_a",
        "url_b",
        "jaccard_distance",
        "jaccard_similarity",
        "match_rank",
    )
)

FINAL_PARQUET_PATH.mkdir(parents=True, exist_ok=True)
DUPLICATE_AUDIT_PATH.mkdir(parents=True, exist_ok=True)

(
    final_docs
    .write
    .mode("overwrite")
    .partitionBy("crawl_date")
    .parquet(str(FINAL_PARQUET_PATH))
)

(
    duplicate_audit
    .write
    .mode("overwrite")
    .parquet(str(DUPLICATE_AUDIT_PATH))
)

print("final_docs rows:", final_docs.count())
print("wrote parquet to:", FINAL_PARQUET_PATH)
print("wrote duplicate audit to:", DUPLICATE_AUDIT_PATH)
final_docs.orderBy("crawl_date", "source_file", "url").show(20, truncate=80)

final_docs rows: 1962
wrote parquet to: data/parquet/final_docs
wrote duplicate audit to: data/parquet/duplicate_audit
+---------------+----------+--------------------+------+-------------------------------------------------------+--------------------------------------+--------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+--------+--------------------------------------------------------------------------------+---------------------------------------------------------+------------------+------------------+-------------+
|       crawl_id|crawl_date|crawl_file_timestamp|doc_id|                                            source_file|                                domain|                                                                             url|                                                                            

In [13]:
deduped_docs.count()

1962

## Notes

- Equal MinHash signatures are only a bucket hint, not proof of equality. Use exact text dedup for exact duplicates, and use Jaccard similarity from `approxSimilarityJoin` for near duplicates.
- If you want to scale beyond this local workflow, the next step is to preprocess WET files into a structured format like JSONL or Parquet outside Spark, then run Spark on that structured dataset.